# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [20]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
]
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},
]

**One row of my result is one:** _line item (an individual product item ordered within a specific order)._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [21]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4, f"Expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected total quantity 7, got {lines['qty'].sum()}"

display(lines)

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [22]:

vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']

print("Unmatched Vendor Report:")
for vendor_id in unmatched['vendor_id'].unique():
    unmatched_units = unmatched[unmatched['vendor_id'] == vendor_id]['qty'].sum()
    print(f"Vendor '{vendor_id}' did not match and has {unmatched_units} units attached to it.")

display(joined)

Unmatched Vendor Report:
Vendor 'V-10' did not match and has 1 units attached to it.


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,NaN,NaN,left_only
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [23]:

vendor_totals = joined.groupby('name', dropna=False)['qty'].sum().reset_index()


display(vendor_totals)

,name,qty
0,Hoos Burgers,6
1,NaN,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [24]:

zone_totals = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
print("Sales by Zone:")
display(zone_totals)


Sales by Zone:


,zone,qty
0,A,6
1,NaN,1


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [25]:

processed_rows = []

for order in ragged_json:
    order_id = order.get('order')
    vendor_id = order.get('vendor_id')
    lines_list = order.get('lines', [])

    if not lines_list or not isinstance(lines_list, list):
        processed_rows.append({
            'order': order_id,
            'vendor_id': vendor_id,
            'item': None,
            'qty': None
        })
    else:
        for line in lines_list:
            processed_rows.append({
                'order': order_id,
                'vendor_id': vendor_id,
                'item': line.get('item'),
                'qty': line.get('qty')
            })

ragged_df = pd.DataFrame(processed_rows)
display(ragged_df)

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,5,V-10,None,NaN
2,6,V-01,Fries,NaN


### Q6 — Validate

In [26]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** "One row per line item" is the most important detail. It makes answering questions about individual products, total units sold, and item-specific trends very simple. If I kept it to "one row per order", I would have lists of items inside cells, making standard aggregations, filtering, and joining extremely complex and inefficient.

**b)** A missing quantity represents unknown/missing data (ex. data entry omission), whereas a quantity of zero represents a known value of no units sold. If we conflate them by filling missing values with 0, future statistical models or averages (like mean quantity per order) will be artificially skewed downward, and business decisions will be based on inaccurate assumptions about customer demand.